# Classificação de Dime e Frida: PyTorch + Keras

Notebook unificado para comparar dois fluxos multi-label lado a lado:

- PyTorch com ResNet18
- Keras com VGG16

Os dois fluxos treinam no mesmo recorte do dataset, geram predições multi-label, exibem Grad-CAM da classe mais relevante para cada modelo e terminam com uma inferência interativa na mesma imagem.

In [ ]:
import os
import sys
import random
import tempfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess
from tensorflow.keras.preprocessing.image import load_img, img_to_array

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    hamming_loss,
)

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
TORCH_EPOCHS = 4
KERAS_EPOCHS = 4
THRESHOLD = 0.5
CLASS_NAMES = ["Dime", "Frida"]
REPO_URL = "https://github.com/PedroM2626/Pet-Classifier-Dime-Frida.git"
DATASET_FOLDER = "my_precious-dataset"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

tf.random.set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch device: {DEVICE}")
print(f"TensorFlow device count: {len(tf.config.list_physical_devices('GPU'))} GPU(s)")

if "google.colab" in sys.modules:
    print("Colab detectado. O dataset pode ser baixado do repositório se ainda não existir.")
else:
    print("Ambiente local detectado.")

## 1. Dataset, rótulos e splits compartilhados

As duas abordagens usam os mesmos caminhos, os mesmos splits e o mesmo conjunto de rótulos multi-label. Como o dataset original está organizado em pastas por classe, cada imagem recebe um vetor multi-hot com duas posições: `Dime` e `Frida`.

In [ ]:
import subprocess
from pathlib import Path


def resolve_dataset_root() -> Path:
    candidates = [
        Path.cwd() / DATASET_FOLDER,
        Path.cwd().parent / DATASET_FOLDER,
        Path.cwd() / "datasets" / DATASET_FOLDER,
        Path("/content") / DATASET_FOLDER,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    if "google.colab" not in sys.modules:
        raise FileNotFoundError(
            f"Não encontrei a pasta '{DATASET_FOLDER}'. Coloque o dataset na raiz do projeto ou execute no Colab para baixar automaticamente."
        )

    repo_dir = Path("Pet-Classifier-Dime-Frida")
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", REPO_URL],
            check=True,
        )
    subprocess.run(
        ["git", "sparse-checkout", "set", DATASET_FOLDER],
        cwd=repo_dir,
        check=True,
    )
    dataset_root = (repo_dir / DATASET_FOLDER).resolve()
    if not dataset_root.exists():
        raise FileNotFoundError(f"O clone foi concluído, mas '{dataset_root}' não existe.")
    return dataset_root


def build_records(dataset_root: Path, class_names: list[str]) -> list[dict]:
    records = []
    valid_suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    for class_index, class_name in enumerate(class_names):
        class_dir = dataset_root / class_name
        if not class_dir.exists():
            continue
        for image_path in sorted(class_dir.rglob("*")):
            if image_path.suffix.lower() in valid_suffixes:
                label = np.zeros(len(class_names), dtype=np.float32)
                label[class_index] = 1.0
                records.append({"path": str(image_path), "label": label})

    if not records:
        raise FileNotFoundError(
            f"Nenhuma imagem válida foi encontrada em '{dataset_root}'. Verifique a estrutura das pastas por classe."
        )

    return records


def safe_split(records, test_size, random_state, stratify_values):
    try:
        return train_test_split(
            records,
            test_size=test_size,
            random_state=random_state,
            stratify=stratify_values,
        )
    except ValueError:
        return train_test_split(
            records,
            test_size=test_size,
            random_state=random_state,
            stratify=None,
        )


dataset_root = resolve_dataset_root()
class_names = [name for name in CLASS_NAMES if (dataset_root / name).exists()]
if len(class_names) < 2:
    class_names = [name for name in CLASS_NAMES if name in CLASS_NAMES]

records = build_records(dataset_root, class_names)
label_indices = [int(np.argmax(record["label"])) for record in records]
train_records, temp_records = safe_split(records, test_size=0.30, random_state=SEED, stratify_values=label_indices)
temp_label_indices = [int(np.argmax(record["label"])) for record in temp_records]
val_records, test_records = safe_split(temp_records, test_size=0.50, random_state=SEED, stratify_values=temp_label_indices)

print(f"Dataset root: {dataset_root}")
print(f"Classes: {class_names}")
print(f"Treino: {len(train_records)} | Validação: {len(val_records)} | Teste: {len(test_records)}")

## 2. Fluxo PyTorch multi-label com ResNet18

Este ramo usa uma ResNet18 pré-treinada como backbone e uma cabeça final com `sigmoid` para prever as duas classes de forma independente. Isso mantém a saída compatível com Grad-CAM e com a comparação direta com o fluxo Keras.

In [ ]:
torch_transform = transforms.Compose(
    [
        transforms.Resize(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)


class MultiLabelImageDataset(Dataset):
    def __init__(self, records, transform=None):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        image = Image.open(record["path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        label = torch.tensor(record["label"], dtype=torch.float32)
        return image, label


def build_loader(records, shuffle):
    dataset = MultiLabelImageDataset(records, transform=torch_transform)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0)


torch_train_loader = build_loader(train_records, shuffle=True)
torch_val_loader = build_loader(val_records, shuffle=False)
torch_test_loader = build_loader(test_records, shuffle=False)


resnet_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, len(class_names))
resnet_model = resnet_model.to(DEVICE)

for parameter in resnet_model.parameters():
    parameter.requires_grad = False
for parameter in resnet_model.layer4.parameters():
    parameter.requires_grad = True
for parameter in resnet_model.fc.parameters():
    parameter.requires_grad = True

resnet_criterion = nn.BCEWithLogitsLoss()
resnet_optimizer = torch.optim.Adam(
    filter(lambda parameter: parameter.requires_grad, resnet_model.parameters()),
    lr=1e-4,
)


def run_torch_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    collected_probs = []
    collected_targets = []

    for images, targets in loader:
        images = images.to(DEVICE)
        targets = targets.to(DEVICE)

        if training:
            optimizer.zero_grad()

        logits = model(images)
        loss = criterion(logits, targets)

        if training:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * images.size(0)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        collected_probs.append(probs)
        collected_targets.append(targets.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    probs = np.vstack(collected_probs)
    targets = np.vstack(collected_targets)
    preds = (probs >= THRESHOLD).astype(int)
    exact_match = accuracy_score(targets, preds)
    return avg_loss, exact_match, targets, probs, preds


for epoch in range(TORCH_EPOCHS):
    train_loss, train_exact, _, _, _ = run_torch_epoch(
        resnet_model, torch_train_loader, resnet_criterion, resnet_optimizer
    )
    val_loss, val_exact, _, _, _ = run_torch_epoch(
        resnet_model, torch_val_loader, resnet_criterion
    )
    print(
        f"[PyTorch] epoch {epoch + 1}/{TORCH_EPOCHS} | "
        f"train_loss={train_loss:.4f} train_exact={train_exact:.4f} | "
        f"val_loss={val_loss:.4f} val_exact={val_exact:.4f}"
    )


torch_test_loss, torch_test_exact, torch_y_true, torch_y_prob, torch_y_pred = run_torch_epoch(
    resnet_model, torch_test_loader, resnet_criterion
)

torch_test_metrics = {
    "loss": torch_test_loss,
    "exact_match": torch_test_exact,
    "hamming_loss": hamming_loss(torch_y_true, torch_y_pred),
    "precision_micro": precision_score(torch_y_true, torch_y_pred, average="micro", zero_division=0),
    "recall_micro": recall_score(torch_y_true, torch_y_pred, average="micro", zero_division=0),
    "f1_micro": f1_score(torch_y_true, torch_y_pred, average="micro", zero_division=0),
    "f1_macro": f1_score(torch_y_true, torch_y_pred, average="macro", zero_division=0),
}

print("\nRelatório PyTorch")
print(classification_report(torch_y_true, torch_y_pred, target_names=class_names, zero_division=0))
print(pd.Series(torch_test_metrics).to_string())

## 3. Fluxo Keras multi-label com VGG16

Este ramo usa a VGG16 como backbone e uma cabeça densa com `sigmoid`, mantendo o mesmo formato multi-label e a mesma divisão de treino, validação e teste.

In [ ]:
def load_keras_split(records):
    images = []
    labels = []
    for record in records:
        pil_image = load_img(record["path"], target_size=IMAGE_SIZE)
        image_array = img_to_array(pil_image)
        image_array = vgg_preprocess(image_array)
        images.append(image_array)
        labels.append(record["label"])
    return np.asarray(images, dtype=np.float32), np.asarray(labels, dtype=np.float32)


keras_train_x, keras_train_y = load_keras_split(train_records)
keras_val_x, keras_val_y = load_keras_split(val_records)
keras_test_x, keras_test_y = load_keras_split(test_records)

keras_base = keras.applications.VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(*IMAGE_SIZE, 3),
)
keras_base.trainable = False

keras_inputs = keras.Input(shape=(*IMAGE_SIZE, 3))
x = keras_base(keras_inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.2)(x)
keras_outputs = layers.Dense(len(class_names), activation="sigmoid")(x)
keras_model = keras.Model(keras_inputs, keras_outputs, name="vgg16_multilabel")

keras_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=[keras.metrics.BinaryAccuracy(name="binary_accuracy")],
)

keras_history = keras_model.fit(
    keras_train_x,
    keras_train_y,
    validation_data=(keras_val_x, keras_val_y),
    epochs=KERAS_EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
)

keras_test_probs = keras_model.predict(keras_test_x, verbose=0)
keras_test_pred = (keras_test_probs >= THRESHOLD).astype(int)
keras_test_loss, keras_test_binary_acc = keras_model.evaluate(
    keras_test_x,
    keras_test_y,
    verbose=0,
)
keras_test_exact = accuracy_score(keras_test_y, keras_test_pred)

keras_test_metrics = {
    "loss": keras_test_loss,
    "binary_accuracy": keras_test_binary_acc,
    "exact_match": keras_test_exact,
    "hamming_loss": hamming_loss(keras_test_y, keras_test_pred),
    "precision_micro": precision_score(keras_test_y, keras_test_pred, average="micro", zero_division=0),
    "recall_micro": recall_score(keras_test_y, keras_test_pred, average="micro", zero_division=0),
    "f1_micro": f1_score(keras_test_y, keras_test_pred, average="micro", zero_division=0),
    "f1_macro": f1_score(keras_test_y, keras_test_pred, average="macro", zero_division=0),
}

print("\nRelatório Keras")
print(classification_report(keras_test_y, keras_test_pred, target_names=class_names, zero_division=0))
print(pd.Series(keras_test_metrics).to_string())

## 4. Comparação e Grad-CAM

Aqui ficam os utilitários compartilhados para escolher a classe de explicação, gerar Grad-CAM em cada framework e montar uma comparação direta entre os dois modelos.

In [ ]:
def choose_cam_index(probabilities, threshold=THRESHOLD):
    positive_indices = np.where(probabilities >= threshold)[0]
    if len(positive_indices) > 0:
        positive_probs = probabilities[positive_indices]
        return int(positive_indices[int(np.argmax(positive_probs))])
    return int(np.argmax(probabilities))


def overlay_heatmap(rgb_image, heatmap, alpha=0.40):
    heatmap = np.clip(heatmap, 0.0, 1.0)
    heatmap_uint8 = np.uint8(255 * heatmap)
    colored_heatmap = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    colored_heatmap = cv2.cvtColor(colored_heatmap, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(rgb_image.astype(np.uint8), 1 - alpha, colored_heatmap, alpha, 0)


def torch_gradcam(model, image_tensor, class_idx, target_layer):
    activations = {}
    gradients = {}

    def forward_hook(_, __, output):
        activations["value"] = output

    def backward_hook(_, __, output):
        gradients["value"] = output[0]

    handle_forward = target_layer.register_forward_hook(forward_hook)
    handle_backward = target_layer.register_full_backward_hook(backward_hook)

    model.zero_grad()
    logits = model(image_tensor)
    score = logits[:, class_idx].sum()
    score.backward()

    handle_forward.remove()
    handle_backward.remove()

    feature_maps = activations["value"].detach()
    grads = gradients["value"].detach()
    weights = grads.mean(dim=(2, 3), keepdim=True)
    heatmap = torch.sum(weights * feature_maps, dim=1).squeeze()
    heatmap = torch.relu(heatmap).cpu().numpy()
    heatmap = cv2.resize(heatmap, IMAGE_SIZE)
    heatmap = heatmap - heatmap.min()
    heatmap = heatmap / (heatmap.max() + 1e-8)
    return heatmap


def keras_gradcam(model, image_array, class_idx, last_conv_layer_name="block5_conv3"):
    grad_model = keras.Model(
        model.inputs,
        [model.get_layer(last_conv_layer_name).output, model.output],
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(image_array)
        class_channel = predictions[:, class_idx]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    heatmap = tf.image.resize(heatmap[..., tf.newaxis], IMAGE_SIZE).numpy().squeeze()
    return heatmap


comparison_table = pd.DataFrame(
    [
        {
            "métrica": "exact_match",
            "PyTorch": torch_test_metrics["exact_match"],
            "Keras": keras_test_metrics["exact_match"],
        },
        {
            "métrica": "hamming_loss",
            "PyTorch": torch_test_metrics["hamming_loss"],
            "Keras": keras_test_metrics["hamming_loss"],
        },
        {
            "métrica": "f1_micro",
            "PyTorch": torch_test_metrics["f1_micro"],
            "Keras": keras_test_metrics["f1_micro"],
        },
        {
            "métrica": "f1_macro",
            "PyTorch": torch_test_metrics["f1_macro"],
            "Keras": keras_test_metrics["f1_macro"],
        },
    ]
)

print("Comparação resumida entre os fluxos")
display(comparison_table)

## 5. Inferência final interativa

Selecione uma imagem, compare as probabilidades dos dois fluxos e veja os dois Grad-CAMs lado a lado para a mesma entrada.

In [ ]:
def select_image_path():
    if "google.colab" in sys.modules:
        from google.colab import files

        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("Nenhuma imagem foi enviada.")
        return next(iter(uploaded.keys()))

    try:
        import tkinter as tk
        from tkinter import filedialog

        root_window = tk.Tk()
        root_window.withdraw()
        selected_path = filedialog.askopenfilename(
            title="Selecione uma imagem",
            filetypes=[("Imagens", "*.jpg *.jpeg *.png *.bmp *.webp")],
        )
        root_window.destroy()
        if selected_path:
            return selected_path
    except Exception:
        pass

    selected_path = input("Digite o caminho completo da imagem: ").strip()
    if not selected_path:
        raise RuntimeError("Nenhum caminho de imagem foi informado.")
    return selected_path


selected_image_path = select_image_path()
selected_pil_image = Image.open(selected_image_path).convert("RGB")
selected_rgb_image = np.array(selected_pil_image.resize(IMAGE_SIZE))

selected_torch_tensor = torch_transform(selected_pil_image).unsqueeze(0).to(DEVICE)
selected_keras_array = img_to_array(selected_pil_image.resize(IMAGE_SIZE)).astype(np.float32)
selected_keras_array = vgg_preprocess(selected_keras_array)
selected_keras_array = np.expand_dims(selected_keras_array, axis=0)

resnet_model.eval()
with torch.no_grad():
    torch_logits = resnet_model(selected_torch_tensor)
    torch_probs = torch.sigmoid(torch_logits).detach().cpu().numpy()[0]

torch_pred_labels = [class_names[i] for i, prob in enumerate(torch_probs) if prob >= THRESHOLD]
if not torch_pred_labels:
    torch_pred_labels = [class_names[int(np.argmax(torch_probs))]]

torch_cam_idx = choose_cam_index(torch_probs)
torch_heatmap = torch_gradcam(resnet_model, selected_torch_tensor, torch_cam_idx, resnet_model.layer4)
torch_overlay = overlay_heatmap(selected_rgb_image, torch_heatmap)

keras_probs = keras_model.predict(selected_keras_array, verbose=0)[0]
keras_pred_labels = [class_names[i] for i, prob in enumerate(keras_probs) if prob >= THRESHOLD]
if not keras_pred_labels:
    keras_pred_labels = [class_names[int(np.argmax(keras_probs))]]

keras_cam_idx = choose_cam_index(keras_probs)
keras_heatmap = keras_gradcam(keras_model, selected_keras_array, keras_cam_idx)
keras_overlay = overlay_heatmap(selected_rgb_image, keras_heatmap)

comparison_df = pd.DataFrame(
    {
        "Classe": class_names,
        "PyTorch": torch_probs,
        "Keras": keras_probs,
    }
)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(selected_rgb_image)
axes[0].set_title("Imagem original")
axes[0].axis("off")

axes[1].imshow(torch_overlay)
axes[1].set_title(f"PyTorch Grad-CAM\n{class_names[torch_cam_idx]}")
axes[1].axis("off")

axes[2].imshow(keras_overlay)
axes[2].set_title(f"Keras Grad-CAM\n{class_names[keras_cam_idx]}")
axes[2].axis("off")

plt.tight_layout()
plt.show()

print("Imagem selecionada:", selected_image_path)
print("PyTorch detectou:", ", ".join(torch_pred_labels))
print("Keras detectou:", ", ".join(keras_pred_labels))
print("\nProbabilidades comparadas")
display(comparison_df)

print("\nComparação textual")
print(f"PyTorch focou em: {class_names[torch_cam_idx]} | Keras focou em: {class_names[keras_cam_idx]}")
print(
    "Resultado final: "
    f"PyTorch -> {', '.join(torch_pred_labels)} | "
    f"Keras -> {', '.join(keras_pred_labels)}"
)